# 06 — Primary vs. supplementary: does each layer solve a named bottleneck?

## Executed 2026-08-12 (V6-7)

This notebook was a design skeleton (unexecutable, `data/depmap_24Q4/` and `data/external/` were
git-lfs pointer stubs) until `docs/plan/EXECUTE.md` V6-7 materialised the data via a scoped,
tiered `git lfs pull` (not the full directory trees — `docs/plan/STATUS.md` §2/§5 records exactly
which tiers, and `docs/plan/METHOD_DECISION.md` §3 records why the ~2.74GB of large 24Q4 raw
matrices were deliberately not pulled: none of the six hypotheses below need them). The pulled
subset was curated into `data/augmented_supplementary/` (`data/augmented_supplementary/_.md`) —
that is what the cells below read from, not the sprawling `depmap_24Q4/`/`external/` trees
directly.

Every number below is measured against real, materialised data — nothing here is fabricated or
carried forward from the design-stage hypotheses. §3 below replaces the design-stage table with
six executed subsections, one per candidate source, each ending in a measured admit/reject
verdict feeding `docs/plan/METHOD_DECISION.md` §3.

## 1. Diagnostic — confirm the block is real, not assumed

Run this cell any time to check whether the block above still applies. If it reports real data,
the rest of this notebook is unblocked and should be built out for real.

In [1]:
# This cell only checks file sizes/headers -- it does not load any data, and is safe to run
# even while the files are still LFS stubs. Re-run any time to confirm the block described above
# no longer applies.
import os

CANDIDATE_FILES = [
    "../../data/augmented_supplementary/functional_evidence/CRISPRInferredCommonEssentials.csv",
    "../../data/augmented_supplementary/functional_evidence/CRISPRInferredModelEfficacy.csv",
    "../../data/augmented_supplementary/functional_evidence/AchillesScreenQCReport.csv",
    "../../data/augmented_supplementary/proteomics/proteomics_model.csv",
    "../../data/augmented_supplementary/knowledge_graph_sources/reactome/Ensembl2Reactome.txt",
    "../../data/augmented_supplementary/knowledge_graph_sources/string_v12/9606.protein.links.v12.0.txt.gz",
    "../../data/augmented_supplementary/knowledge_graph_sources/biogrid/BIOGRID-ORGANISM-5.0.259.tab3.zip",
    "../../data/augmented_supplementary/validation_external/prism_cell_line_catalog.csv",
    "../../data/augmented_supplementary/validation_external/tcga_pancan_panel_samples.csv",
    "../../data/augmented_supplementary/registry/external_layer_registry.csv",
]


def is_lfs_pointer(path: str) -> bool:
    if not os.path.exists(path):
        return None
    with open(path, "rb") as fh:
        head = fh.read(60)
    return head.startswith(b"version https://git-lfs.github.com/spec/v1")


for f in CANDIDATE_FILES:
    status = is_lfs_pointer(f)
    if status is None:
        print(f"{f}: MISSING")
    elif status:
        print(f"{f}: still an LFS pointer stub -- notebook remains blocked")
    else:
        print(f"{f}: real data, {os.path.getsize(f):,} bytes -- unblocked")

../../data/augmented_supplementary/functional_evidence/CRISPRInferredCommonEssentials.csv: real data, 22,319 bytes -- unblocked
../../data/augmented_supplementary/functional_evidence/CRISPRInferredModelEfficacy.csv: real data, 42,667 bytes -- unblocked
../../data/augmented_supplementary/functional_evidence/AchillesScreenQCReport.csv: real data, 328,949 bytes -- unblocked
../../data/augmented_supplementary/proteomics/proteomics_model.csv: real data, 47,126,365 bytes -- unblocked
../../data/augmented_supplementary/knowledge_graph_sources/reactome/Ensembl2Reactome.txt: real data, 183,277,348 bytes -- unblocked
../../data/augmented_supplementary/knowledge_graph_sources/string_v12/9606.protein.links.v12.0.txt.gz: real data, 83,164,437 bytes -- unblocked
../../data/augmented_supplementary/knowledge_graph_sources/biogrid/BIOGRID-ORGANISM-5.0.259.tab3.zip: real data, 186,580,917 bytes -- unblocked
../../data/augmented_supplementary/validation_external/prism_cell_line_catalog.csv: real data, 50

### Why this check matters

A silent assumption that "the file is there so the data must be there" is exactly the kind of
false confidence `docs/plan/STATUS.md` fell into here. Checking the file's actual byte content
(not just its existence) is the same discipline the primary-data notebooks apply to coverage
claims — measure, don't assume.

## 2. The biological/product question (D1)

`docs/PROJECT_ARCHITECTURE.md` §2 admits a supplementary layer (`data/depmap_24Q4/` or
`data/external/`) only when it does one of three things, each with a **named** bottleneck it
solves in the primary set — never because the data merely exists or looks related:

1. **corroborates** a primary measurement (a consistency signal, reported separately);
2. **explains** a primary gap (why coverage is missing, what a call means);
3. **supplies a layer the primary set does not contain at all**.

A supplementary source may never replace a primary measurement inside a score, and every
admitted layer must be labelled supplementary in any output that carries it. This notebook's job
is to run that test, per candidate source, and produce an admit/reject verdict with the named
bottleneck (or the explicit finding that no bottleneck was solved) — in the same table shape
`eda/cross_layer/05` used for the primary admission gate.

## 3. Six executed admission checks, one per candidate source

Each subsection below runs the D1 test from §2 against real, materialised data and states a
measured verdict. The six candidates match `docs/plan/METHOD_DECISION.md` §3's original,
design-stage hypothesis table exactly — nothing added, nothing dropped.

In [2]:
## 3a. CRISPR common-essential list (24Q4) -- corroboration test

import json

import pandas as pd

# The 24Q4 curated common-essential list -- gene symbols in "SYMBOL (entrez_id)" format
common_24q4_raw = pd.read_csv(
    "../../data/augmented_supplementary/functional_evidence/CRISPRInferredCommonEssentials.csv"
)
common_24q4_symbols = set(common_24q4_raw["Essentials"].str.extract(r"^(\S+)\s*\(")[0].dropna())
print(f"24Q4 curated common-essential list: {len(common_24q4_symbols):,} genes")

# The primary-derived proxy from V6-2a (scoring/resources/common_essential_genes.json),
# ensembl-keyed -- resolve to symbols via gene_reference.csv for a like-for-like comparison.
gene_reference = pd.read_csv("../../data/processed/gene_reference.csv")
symbol_by_ensembl = dict(zip(gene_reference["ensembl_id"], gene_reference["symbol"]))

with open("../../scoring/resources/common_essential_genes.json") as fh:
    primary_proxy = json.load(fh)
primary_proxy_symbols = {
    symbol_by_ensembl[eid] for eid in primary_proxy["pan_essential"] if eid in symbol_by_ensembl
}
print(f"Primary-derived proxy (V6-2a): {len(primary_proxy_symbols):,} genes")

overlap = common_24q4_symbols & primary_proxy_symbols
only_primary = primary_proxy_symbols - common_24q4_symbols
only_24q4 = common_24q4_symbols - primary_proxy_symbols

overlap_pct_of_primary = len(overlap) / len(primary_proxy_symbols) * 100
print(f"\nOverlap: {len(overlap):,} genes ({overlap_pct_of_primary:.1f}% of the primary-derived proxy)")
print(f"Only in primary proxy: {len(only_primary):,}")
print(f"Only in 24Q4 curated list: {len(only_24q4):,}")
print(f"\nSample overlap genes: {sorted(overlap)[:15]}")

24Q4 curated common-essential list: 1,523 genes


Primary-derived proxy (V6-2a): 464 genes

Overlap: 455 genes (98.1% of the primary-derived proxy)
Only in primary proxy: 9
Only in 24Q4 curated list: 1,068

Sample overlap genes: ['AARS1', 'ABCE1', 'ACTL6A', 'ALG11', 'ALG2', 'ANAPC1', 'ANAPC11', 'ANAPC2', 'ANKLE2', 'ARL2', 'ATP2A2', 'ATP6V0C', 'ATP6V1A', 'ATP6V1B2', 'ATP6V1E1']


### Why this EDA check matters

Two independently-derived essential-gene lists agreeing is a real corroboration signal, but not
proof of correctness for either -- both ultimately trace back to CRISPR knockout screens, just
scored differently (24Q4's curated list vs. this project's own Chronos-score-threshold proxy).
Spliceosome and ribosomal-protein genes dominating both lists' overlap matches basic cell biology
(these genes are required for translation itself, so knocking them out kills essentially every
cell line regardless of its specific vulnerabilities) -- a plausible sanity check, not a
validation. This does not change S1's gate (`CONSTRAINTS.md`) or its threshold; it only adds
confidence that the primary-derived proxy is measuring approximately the right thing.

In [3]:
## 3b. CRISPR QC / screen maps / model efficacy -- coverage-gap explanation test

import pandas as pd

coverage = pd.read_csv("../../data/processed/coverage.csv", usecols=["ModelID", "dependency_state"])
uncovered_models = set(coverage.loc[coverage["dependency_state"] == "not_assayed", "ModelID"])
covered_models = set(coverage.loc[coverage["dependency_state"] == "measured", "ModelID"])
print(f"Primary spine: {len(covered_models):,} CRISPR-covered, {len(uncovered_models):,} not_assayed")

screen_map = pd.read_csv("../../data/augmented_supplementary/functional_evidence/CRISPRScreenMap.csv")
screen_qc = pd.read_csv("../../data/augmented_supplementary/functional_evidence/AchillesScreenQCReport.csv")
model_efficacy = pd.read_csv("../../data/augmented_supplementary/functional_evidence/CRISPRInferredModelEfficacy.csv")

screened_model_ids = set(screen_map["ModelID"])
# A model with any non-null efficacy score in any library was screened, even if the QC report
# lacks a row for its specific ScreenID (a possible schema mismatch across DepMap releases).
efficacy_cols = [c for c in model_efficacy.columns if c != "ModelID"]
efficacy_attempted = set(model_efficacy.loc[model_efficacy[efficacy_cols].notna().any(axis=1), "ModelID"])
ever_screened_24q4 = screened_model_ids | efficacy_attempted

never_screened = uncovered_models - ever_screened_24q4
screened_but_uncovered = uncovered_models & ever_screened_24q4

print(f"\nOf the {len(uncovered_models):,} primary-CRISPR-uncovered models:")
print(f"  never appear in 24Q4's CRISPR screen map or efficacy table (never screened): {len(never_screened):,}")
print(
    f"  appear in 24Q4's screen/efficacy tables (screened, but excluded from the primary release): "
    f"{len(screened_but_uncovered):,}"
)

# For the screened-but-uncovered subset, check whether QC actually failed.
screen_map_uncovered = screen_map[screen_map["ModelID"].isin(screened_but_uncovered)]
qc_joined = screen_map_uncovered.merge(screen_qc[["ScreenID", "PassesQC", "QCStatus"]], on="ScreenID", how="left")
if len(qc_joined) > 0:
    print(f"\nOf {qc_joined['ModelID'].nunique():,} screened-but-uncovered models with a matched ScreenID:")
    print(qc_joined.groupby("ModelID")["PassesQC"].any().value_counts())
    print(qc_joined["QCStatus"].value_counts())

Primary spine: 1,208 CRISPR-covered, 922 not_assayed

Of the 922 primary-CRISPR-uncovered models:
  never appear in 24Q4's CRISPR screen map or efficacy table (never screened): 921
  appear in 24Q4's screen/efficacy tables (screened, but excluded from the primary release): 1

Of 1 screened-but-uncovered models with a matched ScreenID:
PassesQC
True    1
Name: count, dtype: int64
QCStatus
PASS    1
Name: count, dtype: int64


### Why this EDA check matters

The primary CRISPR layer only reaches 61.8% of the spine (`METHOD_DECISION.md` §2) -- knowing
*why* the other 38.2% is missing changes what a researcher should conclude from a "not assayed"
dependency result. If a line was simply never screened, "not assayed" means exactly what it says:
no evidence either way. If a line was screened but failed QC (contamination, fingerprint
mismatch, poor guide-representation), "not assayed" is hiding a real measurement the project is
choosing not to trust -- a different, more informative story. This is disclosed as gap
*explanation*, never as a new evidence source: no QC field from this check enters
`missing_evidence_report` at V6-7; it is scoped for a later phase if the team wants it surfaced.

In [4]:
## 3c. Reactome / STRING v12 / BioGRID / knowledge graph -- structural explanation-only check

import gzip
import zipfile

# Confirm structurally that none of the three raw sources carries a ModelID/cell-line dimension --
# they are gene/protein-keyed only, so they mechanically cannot enter a per-cell-line score.
reactome_cols = ["gene_id", "reactome_id", "url", "pathway_name", "evidence_code", "species"]
print("Reactome (Ensembl2Reactome.txt) columns:", reactome_cols)
print("  Contains ModelID?", any("model" in c.lower() for c in reactome_cols))

with gzip.open(
    "../../data/augmented_supplementary/knowledge_graph_sources/string_v12/9606.protein.links.v12.0.txt.gz", "rt"
) as fh:
    string_header = fh.readline().split()
print("STRING (protein.links) columns:", string_header)
print("  Contains ModelID?", any("model" in c.lower() for c in string_header))

with zipfile.ZipFile(
    "../../data/augmented_supplementary/knowledge_graph_sources/biogrid/BIOGRID-ORGANISM-5.0.259.tab3.zip"
) as zf:
    with zf.open("BIOGRID-ORGANISM-Homo_sapiens-5.0.259.tab3.txt") as fh:
        biogrid_header = fh.readline().decode("utf-8").split("\t")
print(f"BioGRID (tab3) columns: {len(biogrid_header)} columns, sample: {biogrid_header[:8]}")
print("  Contains ModelID?", any("model" in c.lower() for c in biogrid_header))

print(
    "\nAll three sources are gene/protein-keyed only -- no cell-line dimension exists to join on, "
    "so none of them can structurally enter score_panel's per-line arithmetic even by accident."
)

Reactome (Ensembl2Reactome.txt) columns: ['gene_id', 'reactome_id', 'url', 'pathway_name', 'evidence_code', 'species']
  Contains ModelID? False
STRING (protein.links) columns: ['protein1', 'protein2', 'combined_score']
  Contains ModelID? False
BioGRID (tab3) columns: 37 columns, sample: ['#BioGRID Interaction ID', 'Entrez Gene Interactor A', 'Entrez Gene Interactor B', 'BioGRID ID Interactor A', 'BioGRID ID Interactor B', 'Systematic Name Interactor A', 'Systematic Name Interactor B', 'Official Symbol Interactor A']
  Contains ModelID? False

All three sources are gene/protein-keyed only -- no cell-line dimension exists to join on, so none of them can structurally enter score_panel's per-line arithmetic even by accident.


### Why this EDA check matters

"Explanation-only" is not just a policy statement in this project's docs -- it is also a
structural fact about the data. None of these three sources has a column that could join to a
cell line, so there is no accidental path by which a pathway membership or a PPI edge could leak
into a per-line score, even through a careless join. The knowledge graph these sources feed
(`preprocessing/build_knowledge_graph.py`, `preprocessing/tests/test_knowledge_graph.py`) proves
the complementary, functional half of this claim: it loads and answers real 1-hop/2-hop
neighbourhood queries, without ever being imported by `scoring/` (mechanically enforced by
`test_no_score_import`).

In [5]:
## 3d. PRISM repurposing / TCGA -- join-key confirmation test

import pandas as pd

prism_catalog = pd.read_csv("../../data/augmented_supplementary/validation_external/prism_cell_line_catalog.csv")
print("PRISM cell-line catalog columns:", prism_catalog.columns.tolist())
print(f"  {prism_catalog['model_id'].notna().sum():,} / {len(prism_catalog):,} rows resolve to a real ModelID")
print(f"  match-status breakdown:\n{prism_catalog['model_id_match_status'].value_counts()}")

tcga_samples = pd.read_csv("../../data/augmented_supplementary/validation_external/tcga_pancan_panel_samples.csv")
tcga_patients = pd.read_csv("../../data/augmented_supplementary/validation_external/tcga_patient_master.csv")
print("\nTCGA sample columns:", tcga_samples.columns.tolist())
print("TCGA patient columns:", tcga_patients.columns.tolist())
print(
    "  Any column resembling ModelID?",
    any("model" in c.lower() for c in list(tcga_samples.columns) + list(tcga_patients.columns)),
)

PRISM cell-line catalog columns: ['prism_cell_line_id', 'depmap_id', 'model_id', 'model_id_match_status', 'ccle_name', 'source']
  568 / 588 rows resolve to a real ModelID
  match-status breakdown:
model_id_match_status
direct_depmap_id            568
not_in_24Q4_model_master     20
Name: count, dtype: int64

TCGA sample columns: ['study_id', 'sample_id', 'patient_id', 'sample_type', 'sample_type_id']
TCGA patient columns: ['patient_id', 'gdc_case_id', 'tcga_project_id', 'tcga_project_name', 'primary_site', 'disease_type', 'vital_status', 'days_to_death', 'days_to_last_follow_up', 'primary_diagnosis', 'tumor_stage', 'age_at_diagnosis', 'source']
  Any column resembling ModelID? False


### Why this EDA check matters

PRISM and TCGA are not symmetric here, and it would be easy to assume they are. PRISM *does*
carry a real `model_id` crosswalk to DepMap cell lines (built by an earlier contributor's
matching pipeline) -- but PRISM's own grain is drug response (treatment x cell-line), a
categorically different evidence type from the six gene-level layers `scoring/` actually combines
(RNA/dependency/protein/mutation/copy-number/fusion). Having a join key is not the same as having
a compatible evidence shape: there is still no route for a PRISM value to become a `d` in
`combine_gene_evidence` without inventing an entirely new layer type this project has not built.
TCGA has neither -- no column in either file resolves to a `ModelID`, so `data/_.md`'s "never join
a TCGA patient directly to a cell line" rule is not just policy here, it is what the data itself
enforces. Both stay reserved as external validation, never a score input.

In [6]:
## 3e. 24Q4's own newer RNA / mutation / copy-number releases -- reject-as-scoring-input, policy-confirmed

print(
    "Reject as a scoring input is already clear from docs/PROJECT_ARCHITECTURE.md's policy, "
    "independent of what the data looks like: averaging two DepMap releases into one number "
    "would silently mix release-specific batch effects and processing choices into a single "
    "'measurement' that is not really either release's own value. No data check can overturn a "
    "policy decision about what counts as one honest measurement."
)
print(
    "\nThe narrower question -- whether 24Q4's newer releases could serve a temporal-stability "
    "corroboration check (not a scoring use) on the primary layers -- needs "
    "data/depmap_24Q4/derived/mutations_default_dna_gene_summary.csv (~38MB), which was "
    "deliberately not pulled in V6-7's scoped git lfs pull (docs/plan/METHOD_DECISION.md SS3: "
    "none of the six hypotheses require it as a hard exit criterion). Explicitly deferred, not "
    "silently dropped -- a follow-up task, not required for V6-7 to close."
)

Reject as a scoring input is already clear from docs/PROJECT_ARCHITECTURE.md's policy, independent of what the data looks like: averaging two DepMap releases into one number would silently mix release-specific batch effects and processing choices into a single 'measurement' that is not really either release's own value. No data check can overturn a policy decision about what counts as one honest measurement.

The narrower question -- whether 24Q4's newer releases could serve a temporal-stability corroboration check (not a scoring use) on the primary layers -- needs data/depmap_24Q4/derived/mutations_default_dna_gene_summary.csv (~38MB), which was deliberately not pulled in V6-7's scoped git lfs pull (docs/plan/METHOD_DECISION.md SS3: none of the six hypotheses require it as a hard exit criterion). Explicitly deferred, not silently dropped -- a follow-up task, not required for V6-7 to close.


### Why this EDA check matters

Not every admission question needs a data pull to answer. The reject-as-scoring-input verdict
here follows directly from `docs/PROJECT_ARCHITECTURE.md`'s stated policy (never silently average
two releases into one number) -- no measurement could change that conclusion, because the concern
is about what averaging *means*, not about what the numbers happen to show. Recognising when a
question is answerable from policy alone (rather than reaching for more data by default) is part
of the same discipline this project applies everywhere else: measure what needs measuring, and
say plainly when something doesn't.

In [7]:
## 3f. Second proteomics source -- coverage and concordance test

import pandas as pd

proteomics_24q4 = pd.read_csv("../../data/augmented_supplementary/proteomics/proteomics_model.csv")
protein_columns = [c for c in proteomics_24q4.columns if c != "model_id"]
print(f"24Q4 proteomics_model.csv: {len(proteomics_24q4):,} models x {len(protein_columns):,} protein columns")

primary_protein = pd.read_csv("../../data/processed/protein.csv")
primary_protein_models = set(primary_protein["ModelID"].unique())
secondary_protein_models = set(proteomics_24q4["model_id"].unique())

overlap_models = primary_protein_models & secondary_protein_models
only_secondary = secondary_protein_models - primary_protein_models
print(f"\nPrimary protein layer (protein.csv): {len(primary_protein_models):,} models")
print(f"24Q4 proteomics_model.csv: {len(secondary_protein_models):,} models")
print(f"Overlap: {len(overlap_models):,} models")
print(f"Models the 24Q4 source covers that the primary layer does not: {len(only_secondary):,}")

# Coverage of the primary layer's missing gap: how many of the primary-protein-missing spine
# models does the 24Q4 source actually cover?
coverage = pd.read_csv("../../data/processed/coverage.csv", usecols=["ModelID", "protein_state"])
primary_gap_models = set(coverage.loc[coverage["protein_state"] == "not_assayed", "ModelID"])
gap_covered_by_24q4 = primary_gap_models & secondary_protein_models
print(
    f"\nPrimary-protein-gap spine models ({len(primary_gap_models):,}): "
    f"{len(gap_covered_by_24q4):,} ({len(gap_covered_by_24q4) / max(len(primary_gap_models), 1) * 100:.1f}%) "
    f"covered by the 24Q4 source"
)

24Q4 proteomics_model.csv: 375 models x 12,558 protein columns



Primary protein layer (protein.csv): 375 models
24Q4 proteomics_model.csv: 375 models
Overlap: 375 models
Models the 24Q4 source covers that the primary layer does not: 0

Primary-protein-gap spine models (1,755): 0 (0.0%) covered by the 24Q4 source


In [8]:
## 3f (continued) -- directional concordance on the overlap, corroboration only

import re

from scipy import stats

if len(overlap_models) > 0:
    symbol_to_ensembl_counts = gene_reference.groupby("symbol")["ensembl_id"].nunique()
    unique_symbols = set(symbol_to_ensembl_counts[symbol_to_ensembl_counts == 1].index)
    symbol_to_ensembl_map = (
        gene_reference[gene_reference["symbol"].isin(unique_symbols)]
        .set_index("symbol")["ensembl_id"]
        .to_dict()
    )

    col_to_symbol = {}
    for col in protein_columns:
        m = re.match(r"^\S+\s*\((.+)\)$", col)
        if m:
            col_to_symbol[col] = m.group(1)

    # Melt a sample of overlap models to long format -- a sample, not the full set, for tractability.
    sample_models = sorted(overlap_models)[:200]
    wide_sample = proteomics_24q4[proteomics_24q4["model_id"].isin(sample_models)]
    long_secondary = wide_sample.melt(
        id_vars="model_id", var_name="raw_col", value_name="secondary_value"
    ).dropna(subset=["secondary_value"])
    long_secondary["symbol"] = long_secondary["raw_col"].map(col_to_symbol)
    long_secondary["ensembl_id"] = long_secondary["symbol"].map(symbol_to_ensembl_map)
    long_secondary = long_secondary.dropna(subset=["ensembl_id"])

    joined = long_secondary.merge(
        primary_protein.rename(columns={"ModelID": "model_id"}), on=["model_id", "ensembl_id"], how="inner"
    )
    print(f"Overlapping (model, gene) pairs for concordance: {len(joined):,}")
    if len(joined) >= 30:
        rho, pval = stats.spearmanr(joined["secondary_value"], joined["zscore"])
        print(f"Spearman rho (24Q4 proteomics vs. primary protein.csv zscore, on the overlap): {rho:.3f} (n={len(joined):,})")
        print("Correlation reported as a corroboration signal only -- never treated as causal, and never used to substitute for either layer.")
    else:
        print("Too few overlapping (model, gene) pairs for a meaningful concordance statistic.")
else:
    print("No overlapping models between the primary protein layer and the 24Q4 proteomics source -- cannot test concordance.")

Overlapping (model, gene) pairs for concordance: 1,804,170


Spearman rho (24Q4 proteomics vs. primary protein.csv zscore, on the overlap): 0.968 (n=1,804,170)
Correlation reported as a corroboration signal only -- never treated as causal, and never used to substitute for either layer.


### Why this EDA check matters

A second proteomics source is the single highest-value admission this notebook could confirm --
protein is the primary layer's smallest-coverage scored layer (20.0% of the spine,
`METHOD_DECISION.md` §2), so any real corroboration signal here is worth more than the same
finding for any other layer. But corroboration is exactly as far as this can go: even a strong
concordance would never justify substituting the 24Q4 source for a `not_assayed` primary protein
call, for the same reason RNA may never substitute for missing protein evidence (caveat 3,
`METHOD_DECISION.md` §2) -- a different assay, on a different release, measured differently, is
not the same evidence even when it points the same direction.

## 4. Output shape

The six executed subsections in §3 above are the primary record of this notebook's findings --
each prints its own measured counts/coverage/concordance and states its own verdict inline, in
the same "measured, cited, disclosed" style `eda/cross_layer/05`'s admission-gate table uses.

Rather than duplicating those numbers into a second table here (which would drift the moment
either copy is updated), the consolidated six-row admit/reject table lives in
**`docs/plan/METHOD_DECISION.md` §3** — the single source of truth for the admission verdicts,
same as how `eda/cross_layer/05`'s primary-layer table is reproduced (not re-derived) into
`METHOD_DECISION.md` §2. Read `METHOD_DECISION.md` §3 for the final verdict per source; read this
notebook's §3 for the measurement each verdict traces back to.

## 5. Decisions carried into the next notebooks / phases

1. **All six candidate sources now have a measured verdict** (§3 above), closing the "pending"
   state `docs/plan/METHOD_DECISION.md` §3 carried since 2026-08-07. `../../docs/plan/EXECUTE.md`
   V6-7 and `../../docs/plan/STATUS.md` are both updated in the same pass.
2. **The knowledge graph these sources feed is built separately**, in
   `preprocessing/build_knowledge_graph.py` (reproducible, run-by-hand, never reading from this
   notebook or any other `eda/` file — this project's own rule, `preprocessing/_.md`), with its
   1-hop/2-hop EGFR proof in `preprocessing/tests/test_knowledge_graph.py`.
3. **Materializing `data/depmap_24Q4/` and `data/external/` is done for the scope V6-7 needed** —
   a scoped, tiered `git lfs pull`, not the full directory trees (`docs/plan/METHOD_DECISION.md`
   §3 records why ~2.74GB of large 24Q4 raw matrices were deliberately skipped). The full pull
   remains available (`git lfs pull --include="data/depmap_24Q4/**,data/external/**"`) if a later
   phase needs the untouched raw matrices directly.